# 4th Down Decision Model — Data Pull

Pulls 10 seasons (2016–2025, regular season + playoffs) of play-by-play data via `nflreadpy`, filters down to 4th-down plays, and builds a clean `decision` label (`go` / `punt` / `field_goal`) for each one.

In [1]:
import sys
!{sys.executable} -m pip install pyarrow


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
import pandas as pd
import nflreadpy as nfl

## Config

In [3]:
SEASONS = list(range(2016, 2026))  # last 10 completed seasons

# Plays that don't represent a real 4th-down decision: penalties/pre-snap
# dead balls (no_play), missing play type, and kneel-outs to run clock.
EXCLUDED_PLAY_TYPES = {"no_play", "qb_kneel", "qb_spike"}

DECISION_MAP = {
    "punt": "punt",
    "field_goal": "field_goal",
    "pass": "go",
    "run": "go",
}

KEEP_COLUMNS = [
    "game_id",
    "season",
    "season_type",
    "week",
    "posteam",
    "defteam",
    "qtr",
    "down",
    "ydstogo",
    "yardline_100",
    "game_seconds_remaining",
    "half_seconds_remaining",
    "score_differential",
    "play_type",
    "decision",
    "field_goal_result",
    "epa",
    "wp",
    "wpa",
    "desc",
]

# "decision" doesn't exist yet at raw-pull time, it gets computed below
RAW_COLUMNS = [c for c in KEEP_COLUMNS if c != "decision"]

## Pull play-by-play and filter to 4th downs

`nflreadpy` returns a polars DataFrame natively. Converting the *entire* raw table (10 seasons x 372 columns, ~484k rows) to pandas in one shot uses a lot of memory (pandas is heavier than polars for wide, string-heavy tables) and can crash the kernel on memory-constrained machines like Codespaces.

Instead, we load and filter one season at a time: filter to 4th downs and select only the columns we need *while still in polars*, then convert just that small slice to pandas. Peak memory only ever holds one season's raw data, not all ten.

In [4]:
print(f"Pulling play-by-play for seasons {SEASONS[0]}-{SEASONS[-1]}...")

season_frames = []
for season in SEASONS:
    raw = nfl.load_pbp([season])          # polars, one season at a time
    raw = raw.filter(raw["down"] == 4)    # cheap columnar filter before converting
    raw = raw[RAW_COLUMNS]                # drop the other ~350 columns we don't need
    season_frames.append(raw.to_pandas())
    print(f"  {season}: {raw.shape[0]:,} 4th-down plays")

pbp = pd.concat(season_frames, ignore_index=True)
pbp.shape

Pulling play-by-play for seasons 2016-2025...
  2016: 4,082 4th-down plays
  2017: 4,233 4th-down plays
  2018: 3,977 4th-down plays
  2019: 4,042 4th-down plays
  2020: 3,832 4th-down plays
  2021: 4,254 4th-down plays
  2022: 4,300 4th-down plays
  2023: 4,490 4th-down plays
  2024: 4,279 4th-down plays
  2025: 4,290 4th-down plays


(41779, 19)

In [5]:
fourth = pbp[~pbp["play_type"].isin(EXCLUDED_PLAY_TYPES)].copy()
fourth = fourth[fourth["play_type"].notna()]

fourth["decision"] = fourth["play_type"].map(DECISION_MAP)
fourth = fourth[fourth["decision"].isin(["punt", "field_goal", "go"])]

fourth = fourth[KEEP_COLUMNS]
fourth = fourth.dropna(
    subset=["ydstogo", "yardline_100", "score_differential", "game_seconds_remaining"]
)
fourth = fourth.reset_index(drop=True)

print(f"4th-down plays kept: {fourth.shape[0]:,} rows, {fourth.shape[1]} columns")

4th-down plays kept: 39,367 rows, 20 columns


## Sanity check

In [6]:
fourth["decision"].value_counts()

decision
punt          22519
field_goal     9790
go             7058
Name: count, dtype: int64

In [7]:
fourth.head(10)

,game_id,season,season_type,week,posteam,defteam,qtr,down,ydstogo,yardline_100,game_seconds_remaining,half_seconds_remaining,score_differential,play_type,decision,field_goal_result,epa,wp,wpa,desc
0,2016_01_BUF_BAL,2016,REG,1,BAL,BUF,1.0,4.0,6.0,71.0,3412.0,1612.0,0.0,punt,punt,NaN,-0.405048,0.471003,0.021992,(11:52) (Punt formation) 4-S.Koch punts 39 yar...
1,2016_01_BUF_BAL,2016,REG,1,BUF,BAL,1.0,4.0,18.0,76.0,3282.0,1482.0,0.0,punt,punt,NaN,0.224181,0.418961,0.014649,(9:42) (Punt formation) 6-C.Schmidt punts 50 y...
2,2016_01_BUF_BAL,2016,REG,1,BUF,BAL,1.0,4.0,11.0,48.0,3030.0,1230.0,0.0,punt,punt,NaN,0.447958,0.475927,-0.007276,(5:30) (Punt formation) 6-C.Schmidt punts 39 y...
3,2016_01_BUF_BAL,2016,REG,1,BAL,BUF,1.0,4.0,15.0,32.0,2741.0,941.0,0.0,field_goal,field_goal,made,1.797426,0.529800,0.061810,"(:41) 9-J.Tucker 50 yard field goal is GOOD, C..."
4,2016_01_BUF_BAL,2016,REG,1,BUF,BAL,2.0,4.0,10.0,64.0,2587.0,787.0,-3.0,punt,punt,NaN,0.480402,0.370587,-0.004405,(13:07) (Punt formation) 6-C.Schmidt punts 49 ...
5,2016_01_BUF_BAL,2016,REG,1,BUF,BAL,2.0,4.0,1.0,1.0,1986.0,186.0,-10.0,run,go,NaN,2.859358,0.228060,0.112149,"(3:06) 25-L.McCoy left guard for 1 yard, TOUCH..."
6,2016_01_BUF_BAL,2016,REG,1,BUF,BAL,3.0,4.0,1.0,37.0,1553.0,1553.0,-3.0,run,go,NaN,2.620210,0.411699,0.092453,(10:53) (Shotgun) 5-T.Taylor scrambles left en...
7,2016_01_BUF_BAL,2016,REG,1,BUF,BAL,3.0,4.0,15.0,31.0,1423.0,1423.0,-3.0,field_goal,field_goal,missed,-3.855208,0.358248,-0.085883,(8:43) (Field Goal formation) 2-D.Carpenter 49...
8,2016_01_BUF_BAL,2016,REG,1,BAL,BUF,3.0,4.0,2.0,53.0,1363.0,1363.0,3.0,punt,punt,NaN,-1.136957,0.675675,-0.033029,(7:43) (Punt formation) 4-S.Koch punts 46 yard...
9,2016_01_BUF_BAL,2016,REG,1,BUF,BAL,3.0,4.0,4.0,55.0,1158.0,1158.0,-3.0,punt,punt,NaN,0.560674,0.314419,-0.019397,(4:18) (Punt formation) 6-C.Schmidt punts 46 y...


## Save

In [8]:
out_parquet = "data/fourth_downs.parquet"
out_csv = "data/fourth_downs.csv"
fourth.to_parquet(out_parquet, index=False)
fourth.to_csv(out_csv, index=False)
print(f"Saved to {out_parquet} and {out_csv}")

Saved to data/fourth_downs.parquet and data/fourth_downs.csv
